# SPORES Clustering Workbench

This notebook is a configurable analysis workbench for SPORES clustering. It keeps the detailed descriptions and diagnostics close to the code, while the reusable logic lives in `src/calliope_nl_analysis`.

Use it to select a decision variable, compare clustering methods, inspect labels, and export small summary tables.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "model_files").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SPORES_DIR = ROOT / "results" / "spores"
OUTPUT_ROOT = ROOT / "outputs" / "clustering_workbench"

In [ ]:
import pandas as pd

from calliope_nl_analysis.spores import list_spore_records, validate_spore_inventory
from calliope_nl_analysis.workbench import (
    PRESETS,
    build_matrix,
    elbow_curve,
    export_workbench_tables,
    fit_all_methods,
    get_preset,
    hierarchical_linkages,
    method_parameter_table,
    nearest_neighbor_distances,
    pca_scores,
    preset_table,
    selected_method_tables,
    validation_metrics,
)
from calliope_nl_analysis.plots import (
    plot_capacity_centroids,
    plot_dendrograms,
    plot_elbow_curve,
    plot_family_heatmap,
    plot_nearest_neighbor_curve,
    plot_pca_scores,
    plot_timeseries_centroids,
    plot_validation_metrics,
)

## Available Presets

Presets are named, reproducible clustering configurations. They define the feature extraction rule, PCA dimension, and method-specific clustering parameters.


In [ ]:
preset_table()

## Choose Analysis

Set `ANALYSIS_PRESET` to one of the preset names above. Set `SELECTED_METHOD` to one of `K-means`, `Hier. (Ward)`, `Hier. (Complete)`, `Hier. (Single)`, or `DBSCAN`.

To tune default values or add a decision variable, edit `src/calliope_nl_analysis/workbench.py` near `PRESETS`: add or update an `AnalysisPreset`, then update `build_matrix(...)` if the variable needs a new aggregation rule. Keep this notebook as the clear control panel for selecting the preset and method.


In [ ]:
ANALYSIS_PRESET = "flow_cap_default"
# ANALYSIS_PRESET = "cost_operation_variable_default"

SELECTED_METHOD = "Hier. (Single)"
# SELECTED_METHOD = "DBSCAN"
EXPORT_RESULTS = True

preset = get_preset(ANALYSIS_PRESET)
OUTPUT_DIR = OUTPUT_ROOT / preset.key
preset


In [ ]:
inventory = validate_spore_inventory(SPORES_DIR)
records = list_spore_records(SPORES_DIR)
paths = [record.path for record in records]

inventory["total_files"], round(inventory["total_size_bytes"] / 1024**3, 2), inventory["missing_or_incomplete"]

## Feature Matrix

For `flow_cap_default`, national technology capacities are used, transmission is removed, and `lost_load`, `import_power`, `export_power`, `demand_power`, and `curtailment` are dropped. For `cost_operation_variable_default`, operation costs are summed nationally per timestep and divided by the 3-hour timestep resolution.


In [ ]:
matrix = build_matrix(paths, preset)
matrix.shape

In [ ]:
matrix.head()

## PCA Check

Each preset applies PCA before clustering. This section shows how much variance is retained by the configured number of components.


In [ ]:
scores_for_clustering, pca_for_clustering, _ = pca_scores(matrix, n_components=preset.pca_components)
explained_variance = pd.Series(
    pca_for_clustering.explained_variance_ratio_,
    index=[f"PC{i + 1}" for i in range(preset.pca_components)],
    name="explained_variance_ratio",
)
explained_variance.to_frame()

In [ ]:
explained_variance.cumsum().rename("cumulative_explained_variance").to_frame()

## Cluster-Number Diagnostics

Use these diagnostics to review the configured parameters: K-means elbow curve, hierarchical dendrograms, and DBSCAN nearest-neighbor distances.


In [ ]:
elbow = elbow_curve(matrix, preset)
plot_elbow_curve(elbow, title=f"{preset.variable}: K-means elbow")

In [ ]:
linkages = hierarchical_linkages(matrix, preset)
plot_dendrograms(
    linkages,
    cut_distances=preset.hierarchical_cuts,
    title=f"{preset.variable}: hierarchical dendrograms",
)

In [ ]:
nearest_neighbors = nearest_neighbor_distances(matrix, preset)
plot_nearest_neighbor_curve(
    nearest_neighbors,
    eps=preset.dbscan_eps,
    title=f"{preset.variable}: DBSCAN nearest-neighbor distances",
)

## Fit And Compare Methods

All methods below use the tuning values configured in the selected preset.


In [ ]:
labels_by_method = fit_all_methods(matrix, preset)
metrics = validation_metrics(matrix, labels_by_method, pca_components=preset.pca_components).round(3)
params = method_parameter_table(preset).join(metrics[["clusters", "noise_points"]], rsuffix="_output")
params

In [ ]:
metrics

In [ ]:
plot_validation_metrics(metrics, title=f"{preset.variable}: validation metrics")

## Inspect Selected Method

Change `SELECTED_METHOD` in the configuration cell to inspect another clustering result without changing the feature matrix or tuning presets.

In [ ]:
labels = labels_by_method[SELECTED_METHOD]
result_tables = selected_method_tables(matrix, labels)
label_counts = labels.value_counts().sort_index().rename("solutions")
label_counts

In [ ]:
family_counts = result_tables["family_counts"]
plot_family_heatmap(family_counts, title=f"{preset.variable}: {SELECTED_METHOD} SPORES allocation")

In [ ]:
cluster_summary = result_tables["cluster_summary"]
if preset.matrix_kind == "timeseries":
    plot_timeseries_centroids(cluster_summary, title=f"{preset.variable}: {SELECTED_METHOD} centroids")
else:
    plot_capacity_centroids(cluster_summary, title=f"{preset.variable}: {SELECTED_METHOD} centroids")

In [ ]:
representatives = result_tables["representatives"]
representatives

In [ ]:
scores_2d, pca_2d, _ = pca_scores(matrix, n_components=2)
pd.Series(
    pca_2d.explained_variance_ratio_,
    index=["PC1", "PC2"],
    name="explained_variance_ratio",
).to_frame()

In [ ]:
plot_pca_scores(scores_2d, labels, title=f"{preset.variable}: {SELECTED_METHOD} labels in PCA space")

## Export Workbench Tables

Exports small CSV summaries for the currently selected preset and method. The large `.nc` files are never copied.

In [ ]:
tables_to_export = {
    "method_parameters": params,
    "validation_metrics": metrics,
    "elbow": elbow,
    "nearest_neighbor_distances": nearest_neighbors,
    "selected_label_counts": label_counts,
    "selected_family_counts": family_counts,
    "selected_cluster_summary": cluster_summary,
    "selected_representatives": representatives,
    "selected_labelled_matrix": result_tables["labelled_matrix"],
}

if EXPORT_RESULTS:
    written = export_workbench_tables(tables_to_export, OUTPUT_DIR / SELECTED_METHOD.replace(" ", "_").replace(".", ""))
else:
    written = []
written